<a href="https://colab.research.google.com/github/harryscheidt/harryscheidt.github.io/blob/main/BUA_451_finalprojectcode_HarryScheidt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Harry Scheidt Final Project

## Exectuive Summary

The goal of the project is to analyze the ecommerce trends for a company called the look and extract useful business insights. To do this, the first step was data exploration and acquisition.




The initial exploration involded calculating total revenue by category. From this I was able to visualize the revenue of each category in a bar chart. This chart allowed for the visualization of the top categories by revenue. By making it interactive I was able to show exact amounts while still retaining visual simplicity.

My next visualization was a scatter plot showing the demographics of the userbase. By showing this, exectuives in the company can get a better idea of the types of people who use the ecommerce platform and adapt the product to their needs and preferences.

I then focused on building a classification model to predict the result of an order (Complete or Returned or Cancelled). I created a new 'delivery_time' part by calculating the difference between order creation and delivery timestamps. The Random Forest Classifier was used for modeling.

This data had a significant weighting towards the majority class so I underfitted the data and set the majority class equal to the minority class. This improved accuaracy tremendously and reduced the overfitting that was happening prior to the reweighting. As a result, the model correctly predicts a return 71% of the time and correctly predicts a completed order 75% of the time. It has a weighted average accuracy of 73%.

Ultimately, this project was a great success and valuable business insights have been provided.

# Setup Code

In [1]:
!pip install pandas-gbq --quiet
!pip install google-cloud-bigquery pandas

In [2]:
# prompt: imprt numpy, pyplot, pandas, seaborn, bigquery
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from google.cloud import bigquery
from google.colab import auth
auth.authenticate_user()

# Construct a BigQuery client object.
projectID = 'fit-heaven-457823-c7'
client = bigquery.Client(project= projectID)
#test the connection
test_query = "SELECT CURRENT_DATE() as today"
test_result= client.query(test_query).result().to_dataframe()
print("Connection successful! Today's date from BigQuery is:", test_result['today'][0])

Connection successful! Today's date from BigQuery is: 2025-04-30


# Exploratory Data Analysis and Plots

In [3]:
# prompt: Use SQL in BigQuery to explore the dataset bigquery-public-data.thelook_ecommerce.  Extract TWO meaningful business-relevant insights. Visualize your findings using Python (Plotly, Seaborn, Matplotlib, etc.).Include at least one interactive visualization (e.g., using Plotly or ipywidgets). Analyze the total revenue from each category by adding the retail_price of each order in the dataset. Make an interactive bar chart to visualize.

import pandas as pd
import pandas_gbq
from google.cloud import bigquery
import plotly.express as px

# Authenticate with Google Cloud (if needed)
# from google.colab import auth
# auth.authenticate_user()

# Construct a BigQuery client object.
client = bigquery.Client()

# Define the SQL query
query = """
SELECT
    category,
    SUM(retail_price) AS total_revenue
  FROM
    `bigquery-public-data`.thelook_ecommerce.products
  GROUP BY
    category
  ORDER BY
    total_revenue DESC
"""


# Execute the query and load results into a Pandas DataFrame
df = pandas_gbq.read_gbq(query, project_id='fit-heaven-457823-c7') # Replace with your actual project ID


# Insight 1: Identify the top-performing product categories by revenue
print("Top-performing product categories by revenue:")
print(df.head(10))

# Insight 2:  Categories with the lowest revenue might need attention
print("\nCategories with lowest revenue (potential areas for improvement):")
print(df.tail(5))


# Interactive bar chart visualization with Plotly
fig = px.bar(df, x='category', y='total_revenue',
             title='Total Revenue by Product Category',
             labels={'total_revenue': 'Total Revenue', 'category': 'Product Category'},
             hover_data=['total_revenue'],
             color='total_revenue') # Color bars by revenue

fig.update_layout(xaxis_title="Product Category", yaxis_title="Total Revenue", title_x=0.5)
fig.show()


Downloading: 100%|██████████|
Top-performing product categories by revenue:
                        category  total_revenue
0              Outerwear & Coats  207344.269836
1                          Jeans  195608.560285
2                       Sweaters  130829.070003
3                           Swim  103952.560129
4  Fashion Hoodies & Sweatshirts  100606.500083
5            Suits & Sport Coats   93524.599897
6                 Sleep & Lounge   87166.440206
7                         Shorts   80783.900231
8                        Dresses   80414.130199
9                      Intimates   79650.540112

Categories with lowest revenue (potential areas for improvement):
               category  total_revenue
21                Socks   18484.560006
22             Leggings   15312.409976
23      Socks & Hosiery   11162.129981
24  Jumpsuits & Rompers    7358.390022
25        Clothing Sets    3139.230003


In [4]:
# Query to get the number of users by age bracket and gender
query = """
SELECT
    CASE
        WHEN age BETWEEN 10 AND 19 THEN '10-19'
        WHEN age BETWEEN 20 AND 29 THEN '20-29'
        WHEN age BETWEEN 30 AND 39 THEN '30-39'
        WHEN age BETWEEN 40 AND 49 THEN '40-49'
        WHEN age BETWEEN 50 AND 59 THEN '50-59'
        WHEN age BETWEEN 60 AND 69 THEN '60-69'
        WHEN age BETWEEN 70 AND 79 THEN '70-79'
        WHEN age BETWEEN 80 AND 89 THEN '80-89'
        WHEN age >= 90 THEN '90+'
        ELSE 'Unknown'
    END AS age_bracket,
    gender,
    COUNT(*) AS num_users
  FROM
    `bigquery-public-data`.thelook_ecommerce.users
  GROUP BY 1, 2
  ORDER BY age_bracket
"""

df_users_by_age = pandas_gbq.read_gbq(query, project_id='fit-heaven-457823-c7')

# Create the scatter plot
fig = px.scatter(df_users_by_age,
                 x='age_bracket',
                 y='num_users',
                 color='gender',
                 title='Number of Users by Age Bracket and Gender',
                 labels={'num_users': 'Number of Users', 'age_bracket': 'Age Bracket'},
                 hover_data=['num_users', 'gender'])

fig.update_layout(xaxis_title="Age Bracket", yaxis_title="Number of Users", title_x=0.5)
fig.show()


# Business Insight and Summary
# Example: Analyze the distribution of users across different age brackets and genders.
# Identify the age groups with the highest user concentration and analyze gender distribution within those groups.
# This could inform marketing strategies, product development, and targeted advertising campaigns.


print("Summary of User Demographics:")

# Calculate total users
total_users = df_users_by_age['num_users'].sum()
print(f"Total users: {total_users}")

# Find the age bracket with the most users
max_users_age_bracket = df_users_by_age.loc[df_users_by_age['num_users'].idxmax()]
print(f"Age bracket with the most users: {max_users_age_bracket['age_bracket']} ({max_users_age_bracket['num_users']} users)")

# Find the age bracket with the fewest users
min_users_age_bracket = df_users_by_age.loc[df_users_by_age['num_users'].idxmin()]
print(f"Age bracket with the fewest users: {min_users_age_bracket['age_bracket']} ({min_users_age_bracket['num_users']} users)")





Downloading: 100%|██████████|


Summary of User Demographics:
Total users: 100000
Age bracket with the most users: 50-59 (8657 users)
Age bracket with the fewest users: 70-79 (791 users)


# Random Forest Model and Results

In [6]:
# prompt: Build a machine learning model from the dataset bigquery-public-data.thelook_ecommerce.order_items that will predict status of the order. status is a categorical variable. You can create a new column that takes the difference between delivered_at and created_ at to show the amount of time it takes for customers to receive their order after they place it.

# Query to get order data with a new 'delivery_time' column
query = """
SELECT
    order_id,
    user_id,
    created_at,
    status,
    product_id,
    delivered_at,
    TIMESTAMP_DIFF(delivered_at, created_at, SECOND) AS delivery_time  -- Calculate delivery time in seconds
  FROM
    `bigquery-public-data`.thelook_ecommerce.order_items
  WHERE delivered_at IS NOT NULL and created_at IS NOT NULL
"""

df = pandas_gbq.read_gbq(query, project_id='fit-heaven-457823-c7')

# Data preprocessing
# Remove rows with missing values in relevant columns
df.dropna(subset=['delivery_time', 'status'], inplace=True)


# --- Filter data to include only "Complete", "Returned", or "Cancelled" ---
df = df[df['status'].apply(lambda x: x in ['Complete', 'Returned', 'Cancelled'])]

# --- Combine "Returned" and "Cancelled" into a single class ---
df['status'] = df['status'].apply(lambda x: 'Cancelled' if x in ['Returned', 'Cancelled'] else x)


# --- Convert 'status' to numerical labels using Label Encoding ---
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['status'] = le.fit_transform(df['status'])


# --- Resample to have 18349 samples of each class ---
from sklearn.utils import resample
df_class_0 = df[df['status'] == 0]
df_class_1 = df[df['status'] == 1]

df_class_0_resampled = resample(df_class_0, replace=True, n_samples=18349, random_state=42)
df_class_1_resampled = resample(df_class_1, replace=True, n_samples=18349, random_state=42)

df_resampled = pd.concat([df_class_0_resampled, df_class_1_resampled])

# --- Class Distribution After Resampling ---
from collections import Counter

class_distribution = Counter(df_resampled['status'])
print("Class Distribution After Resampling:")
for class_label, count in class_distribution.items():
    print(f"Class {class_label}: {count} samples")
# --- End Class Distribution ---



# Features (X) and target (y)
X = df_resampled[['delivery_time', 'user_id']]  # Example: using delivery time as a predictor
y = df_resampled['status']

# Import train_test_split
from sklearn.model_selection import train_test_split # Added this import statement

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model training (using Random Forest Classifier as an example)
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")
print(classification_report(y_test, y_pred))

Downloading: 100%|██████████|
Class Distribution After Resampling:
Class 0: 18349 samples
Class 1: 18349 samples
Accuracy: 0.728882833787466
              precision    recall  f1-score   support

           0       0.71      0.77      0.74      3656
           1       0.75      0.69      0.72      3684

    accuracy                           0.73      7340
   macro avg       0.73      0.73      0.73      7340
weighted avg       0.73      0.73      0.73      7340



I built a model to predict whether an order would be completed or cancelled/returned. The data I used has a significant weighting towards the majority class. To fix this I underfitted the data and set the majority class equal to the minority class. This improved accuaracy tremendously and reduced the overfitting that was happening prior to the reweighting. This model improves to being correct 73% of the time while improving the F1 score. The model correctly predicts a return 71% of the time and correctly predicts a completed order 75% of the time.
This model could be used to plug in an order to see if there will be a high chance of return and plan logistics accordingly.